In [ ]:
import pandas as pd
import numpy as np

### Project Walkthrough

This project aims to build a machine learning model to predict house prices based on various features. Below is a step-by-step explanation of the process:

---

### 1. Initial Data Loading and Exploration

First, we load the dataset and perform an initial exploration to understand its structure, identify missing values, and examine the distribution of different features.

### 2. Data Cleaning - Dropping Unnecessary Columns

Based on initial exploration, certain columns ('area_type', 'availability', 'society', 'balcony') were identified as less relevant for price prediction or having too many missing values. These columns were dropped to simplify the model and improve performance.

### 3. Data Cleaning - Handling Missing Values

Missing values in `location`, `size`, and `bath` columns were addressed. 'location' and 'size' were filled with their respective modes, and 'bath' was filled with the median value. This ensures that all essential features have complete data for model training.

### 4. Feature Engineering

New features were engineered to provide more useful information to the model:

*   **`bhk` (Bedroom, Hall, Kitchen)**: Extracted from the `size` column, representing the number of bedrooms.
*   **`total_sqft` Conversion**: The `total_sqft` column contained range values (e.g., '1200 - 1470'). A function `convertRange` was created to convert these ranges into a single float value by taking the average of the range.
*   **`price_per_sqft`**: Calculated as `price * 100000 / total_sqft`. This feature helps in identifying outliers and normalizing price based on area.

### 5. Outlier Removal

Outliers can significantly affect model performance. Several steps were taken to remove them:

*   **Grouping Rare Locations**: Locations with fewer than 10 entries were grouped into an 'others' category to handle sparsity and improve the effectiveness of one-hot encoding.
*   **`total_sqft` per `bhk` Outliers**: Properties with `total_sqft` less than 300 per `bhk` were considered outliers and removed.
*   **`price_per_sqft` Outliers (Standard Deviation Method)**: A function `remove_outlier_sqft` was applied to each `location` group. It removed properties where `price_per_sqft` fell outside one standard deviation from the mean `price_per_sqft` for that specific location.
*   **`bhk` Outliers (Price Comparison Method)**: A function `bhk_outlier_remover` was implemented to remove instances where a property with a higher number of `bhk` had a lower `price_per_sqft` than a property with `bhk-1` in the same location. This addresses illogical pricing discrepancies.

### 6. Final Data Preparation and Model Training

After cleaning and feature engineering, the data was prepared for model training:

*   **Dropping Helper Columns**: The `size` and `price_per_sqft` columns were dropped as they were used for feature engineering and outlier removal but are not direct features for the model.
*   **Saving Cleaned Data**: The processed data was saved to a CSV file named `Cleaned_data.csv`.
*   **Splitting Data**: The dataset was split into features (`X`) and target (`y`, which is `price`). Then, it was further divided into training and testing sets using `train_test_split`.
*   **Model Pipeline**: A machine learning pipeline was constructed using:
    *   `OneHotEncoder`: To convert the categorical `location` feature into numerical format.
    *   `StandardScaler`: To scale numerical features.
    *   `LinearRegression`: The chosen regression model.
*   **Model Training and Evaluation**: The pipeline was fitted to the training data, predictions were made on the test set, and the `r2_score` was used to evaluate the model's performance.
*   **Model Persistence**: The trained `LinearRegression` model pipeline was saved using `pickle` for future use.

In [ ]:
data=pd.read_csv('/content/sample_data/Bengaluru_House_Data.csv')

In [ ]:
data.head()

,area_type,availability,location,size,society,total_sqft,bath,balcony,price
0,Super built-up Area,19-Dec,Electronic City Phase II,2 BHK,Coomee,1056,2.0,1.0,39.07
1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,Theanmp,2600,5.0,3.0,120.00
2,Built-up Area,Ready To Move,Uttarahalli,3 BHK,NaN,1440,2.0,3.0,62.00
3,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,Soiewre,1521,3.0,1.0,95.00
4,Super built-up Area,Ready To Move,Kothanur,2 BHK,NaN,1200,2.0,1.0,51.00


In [ ]:
data.shape


(13320, 9)

In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13320 entries, 0 to 13319
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   area_type     13320 non-null  object 
 1   availability  13320 non-null  object 
 2   location      13319 non-null  object 
 3   size          13304 non-null  object 
 4   society       7818 non-null   object 
 5   total_sqft    13320 non-null  object 
 6   bath          13247 non-null  float64
 7   balcony       12711 non-null  float64
 8   price         13320 non-null  float64
dtypes: float64(3), object(6)
memory usage: 936.7+ KB


In [ ]:
for column in data.columns:
  print(data[column].value_counts())
  print("*"*20)


area_type
Super built-up  Area    8790
Built-up  Area          2418
Plot  Area              2025
Carpet  Area              87
Name: count, dtype: int64
********************
availability
Ready To Move    10581
18-Dec             307
18-May             295
18-Apr             271
18-Aug             200
                 ...  
16-Oct               1
17-Jan               1
16-Nov               1
16-Jan               1
14-Jul               1
Name: count, Length: 81, dtype: int64
********************
location
Whitefield                         540
Sarjapur  Road                     399
Electronic City                    302
Kanakpura Road                     273
Thanisandra                        234
                                  ... 
3rd Stage Raja Rajeshwari Nagar      1
Chuchangatta Colony                  1
Electronic City Phase 1,             1
Chikbasavanapura                     1
Abshot Layout                        1
Name: count, Length: 1305, dtype: int64
********************
siz

In [ ]:
data.isna().sum()

,0
area_type,0
availability,0
location,1
size,16
society,5502
total_sqft,0
bath,73
balcony,609
price,0


In [ ]:
data.drop(columns=['area_type','availability','society','balcony'],inplace=True)

In [ ]:
data.describe()


,bath,price
count,13247.000000,13320.000000
mean,2.692610,112.565627
std,1.341458,148.971674
min,1.000000,8.000000
25%,2.000000,50.000000
50%,2.000000,72.000000
75%,3.000000,120.000000
max,40.000000,3600.000000


In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13320 entries, 0 to 13319
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   location    13319 non-null  object 
 1   size        13304 non-null  object 
 2   total_sqft  13320 non-null  object 
 3   bath        13247 non-null  float64
 4   price       13320 non-null  float64
dtypes: float64(2), object(3)
memory usage: 520.4+ KB


In [ ]:
data['location'].value_counts()

,count
location,
Whitefield,540
Sarjapur Road,399
Electronic City,302
Kanakpura Road,273
Thanisandra,234
...,...
3rd Stage Raja Rajeshwari Nagar,1
Chuchangatta Colony,1
"Electronic City Phase 1,",1


In [ ]:
data['location']=data['location'].fillna('Sarjapur Road')

In [ ]:
data['size'].value_counts()

,count
size,
2 BHK,5199
3 BHK,4310
4 Bedroom,826
4 BHK,591
3 Bedroom,547
1 BHK,538
2 Bedroom,329
5 Bedroom,297
6 Bedroom,191


In [ ]:
data['size']=data['size'].fillna('2 BHK')

In [ ]:
data['bath']=data['bath'].fillna(data['bath'].median())

In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13320 entries, 0 to 13319
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   location    13320 non-null  object 
 1   size        13320 non-null  object 
 2   total_sqft  13320 non-null  object 
 3   bath        13320 non-null  float64
 4   price       13320 non-null  float64
dtypes: float64(2), object(3)
memory usage: 520.4+ KB


In [ ]:
data['bhk']=data['size'].str.split().str.get(0).astype(int)

In [ ]:
data[data.bhk>20]

,location,size,total_sqft,bath,price,bhk
1718,2Electronic City Phase II,27 BHK,8000,27.0,230.0,27
4684,Munnekollal,43 Bedroom,2400,40.0,660.0,43


In [ ]:
data['total_sqft'].unique()

array(['1056', '2600', '1440', ..., '1133 - 1384', '774', '4689'],
      dtype=object)

In [ ]:
def convertRange(x):

  temp=x.split('-')
  if len(temp)==2:
    return (float(temp[0])+float(temp[1])/2)
  try:
    return float(x)
  except:
    return None

In [ ]:
data['total_sqft']=data['total_sqft'].apply(convertRange)

In [ ]:
data.head()

,location,size,total_sqft,bath,price,bhk
0,Electronic City Phase II,2 BHK,1056.0,2.0,39.07,2
1,Chikka Tirupathi,4 Bedroom,2600.0,5.0,120.00,4
2,Uttarahalli,3 BHK,1440.0,2.0,62.00,3
3,Lingadheeranahalli,3 BHK,1521.0,3.0,95.00,3
4,Kothanur,2 BHK,1200.0,2.0,51.00,2


In [ ]:
data['price_per_sqft']=data['price']*100000/data['total_sqft']

In [ ]:
data.describe()

,total_sqft,bath,price,bhk,price_per_sqft
count,13274.000000,13320.000000,13320.000000,13320.000000,1.327400e+04
mean,1572.225941,2.688814,112.565627,2.802778,7.883254e+03
std,1254.823072,1.338754,148.971674,1.294496,1.064303e+05
min,1.000000,1.000000,8.000000,1.000000,2.678298e+02
25%,1100.000000,2.000000,50.000000,2.000000,4.227642e+03
50%,1280.000000,2.000000,72.000000,3.000000,5.416667e+03
75%,1690.000000,3.000000,120.000000,3.000000,7.293505e+03
max,52272.000000,40.000000,3600.000000,43.000000,1.200000e+07


In [ ]:
data['location'].value_counts()

,count
location,
Whitefield,540
Sarjapur Road,399
Electronic City,302
Kanakpura Road,273
Thanisandra,234
...,...
Mango Garden Layout,1
Milk Colony,1
"Basnashankari,6th stage,",1


In [ ]:
data['location']=data['location'].apply(lambda x: x.strip())
location_counts=data['location'].value_counts()

In [ ]:
location_counts

,count
location,
Whitefield,541
Sarjapur Road,399
Electronic City,304
Kanakpura Road,273
Thanisandra,237
...,...
Xavier Layout,1
Ramanagara Channapatna,1
Maheswari Nagar,1


In [ ]:
location_count_less_10=location_counts[location_counts<10]
location_count_less_10

,count
location,
4th Block Koramangala,9
Gollahalli,9
Chandra Layout,9
Medahalli,9
Mathikere,9
...,...
Xavier Layout,1
Ramanagara Channapatna,1
Maheswari Nagar,1


In [ ]:
data['location']=data['location'].apply(lambda x : 'others' if x in location_count_less_10 else x)

In [ ]:
data['location'].value_counts()


,count
location,
others,2756
Whitefield,541
Sarjapur Road,399
Electronic City,304
Kanakpura Road,273
...,...
Sector 1 HSR Layout,10
1st Block Koramangala,10
Gunjur Palya,10


In [ ]:
data.describe()

,total_sqft,bath,price,bhk,price_per_sqft
count,13274.000000,13320.000000,13320.000000,13320.000000,1.327400e+04
mean,1572.225941,2.688814,112.565627,2.802778,7.883254e+03
std,1254.823072,1.338754,148.971674,1.294496,1.064303e+05
min,1.000000,1.000000,8.000000,1.000000,2.678298e+02
25%,1100.000000,2.000000,50.000000,2.000000,4.227642e+03
50%,1280.000000,2.000000,72.000000,3.000000,5.416667e+03
75%,1690.000000,3.000000,120.000000,3.000000,7.293505e+03
max,52272.000000,40.000000,3600.000000,43.000000,1.200000e+07


In [ ]:
(data['total_sqft']/data['bhk']).describe()

,0
count,13274.000000
mean,579.732987
std,392.247238
min,0.250000
25%,475.000000
50%,554.000000
75%,627.500000
max,26136.000000


In [ ]:
data=data[((data['total_sqft']/data['bhk'])>=300)]
data.describe()

,total_sqft,bath,price,bhk,price_per_sqft
count,12530.000000,12530.000000,12530.000000,12530.000000,12530.000000
mean,1607.911902,2.559537,111.382401,2.650838,6278.292738
std,1277.977557,1.077938,152.077329,0.976678,4171.053803
min,300.000000,1.000000,8.440000,1.000000,267.829813
25%,1120.000000,2.000000,49.000000,2.000000,4177.777778
50%,1302.500000,2.000000,70.000000,3.000000,5273.281068
75%,1710.000000,3.000000,115.000000,3.000000,6896.551724
max,52272.000000,16.000000,3600.000000,16.000000,176470.588235


In [ ]:
data.shape

(12530, 7)

In [ ]:
def remove_outlier_sqft(df):
  df_output=pd.DataFrame()
  for key,subdf in df.groupby('location'):
    m=np.mean(subdf.price_per_sqft)
    st=np.std(subdf.price_per_sqft)
    gen_df=subdf[(subdf.price_per_sqft>(m-st)) & (subdf.price_per_sqft<=(m+st))]
    df_output=pd.concat([df_output,gen_df],ignore_index=True)
  return df_output

data=remove_outlier_sqft(data)
data.describe()

,total_sqft,bath,price,bhk,price_per_sqft
count,10268.000000,10268.000000,10268.000000,10268.000000,10268.000000
mean,1519.480630,2.474094,91.738232,2.577328,5651.580450
std,906.125933,0.982664,88.512715,0.898339,2297.281833
min,300.000000,1.000000,10.000000,1.000000,1250.000000
25%,1110.000000,2.000000,49.000000,2.000000,4210.526316
50%,1290.000000,2.000000,67.000000,2.000000,5163.188406
75%,1651.000000,3.000000,100.000000,3.000000,6421.803141
max,30400.000000,16.000000,2200.000000,16.000000,24509.803922


In [ ]:
def bhk_outlier_remover(df):
  exclude_indices=np.array([])
  for location , location_df in df.groupby('location'):
    bhk_stats={}
    for bhk , bhk_df in location_df.groupby('bhk'):
      bhk_stats[bhk]={
          'mean':np.mean(bhk_df.price_per_sqft),
          'std':np.std(bhk_df.price_per_sqft),
          'count':bhk_df.shape[0]
      }
    print(location,bhk_stats)


    for bhk, bhk_df in location_df.groupby('bhk'):
      stats=bhk_stats.get(bhk-1)
      if stats and stats['count']>5:
        exclude_indices=np.append(exclude_indices,bhk_df[bhk_df.price_per_sqft<(stats['mean'])].index.values)
  return df.drop(exclude_indices,axis='index')



In [ ]:
data=bhk_outlier_remover(data)


1st Block Jayanagar {2: {'mean': np.float64(11983.805668016194), 'std': 0.0, 'count': 1}, 3: {'mean': np.float64(11756.16905248807), 'std': 701.6243657657865, 'count': 3}, 4: {'mean': np.float64(15018.711280365416), 'std': 1.2278182423353805, 'count': 3}}
1st Block Koramangala {2: {'mean': np.float64(7695.065329936724), 'std': 78.78626016928274, 'count': 2}, 3: {'mean': np.float64(8936.170212765957), 'std': 0.0, 'count': 1}, 4: {'mean': np.float64(13333.333333333334), 'std': 3333.333333333334, 'count': 2}}
1st Phase JP Nagar {1: {'mean': np.float64(5952.380952380952), 'std': 0.0, 'count': 1}, 2: {'mean': np.float64(7931.806799837383), 'std': 1534.1422783514054, 'count': 8}, 3: {'mean': np.float64(9151.192151725822), 'std': 1054.731726021645, 'count': 7}, 4: {'mean': np.float64(7537.92218148637), 'std': 1607.0591069513537, 'count': 3}, 5: {'mean': np.float64(5666.666666666667), 'std': 0.0, 'count': 1}}
2nd Phase Judicial Layout {2: {'mean': np.float64(3851.8518518518517), 'std': 497.593

In [ ]:
data.shape

(7397, 7)

In [ ]:
data

,location,size,total_sqft,bath,price,bhk,price_per_sqft
0,1st Block Jayanagar,4 BHK,2850.0,4.0,428.0,4,15017.543860
1,1st Block Jayanagar,3 BHK,1630.0,3.0,194.0,3,11901.840491
2,1st Block Jayanagar,3 BHK,1875.0,2.0,235.0,3,12533.333333
3,1st Block Jayanagar,3 BHK,1200.0,2.0,130.0,3,10833.333333
4,1st Block Jayanagar,2 BHK,1235.0,2.0,148.0,2,11983.805668
...,...,...,...,...,...,...,...
10259,others,2 BHK,1200.0,2.0,70.0,2,5833.333333
10260,others,1 BHK,1800.0,1.0,200.0,1,11111.111111
10263,others,2 BHK,1353.0,2.0,110.0,2,8130.081301
10264,others,1 Bedroom,812.0,1.0,26.0,1,3201.970443


In [ ]:
data.drop(columns=['size','price_per_sqft'],inplace=True)

**CLEANED DATA**

In [ ]:
data.head()

,location,total_sqft,bath,price,bhk
0,1st Block Jayanagar,2850.0,4.0,428.0,4
1,1st Block Jayanagar,1630.0,3.0,194.0,3
2,1st Block Jayanagar,1875.0,2.0,235.0,3
3,1st Block Jayanagar,1200.0,2.0,130.0,3
4,1st Block Jayanagar,1235.0,2.0,148.0,2


In [ ]:
data.to_csv("Cleaned_data.csv")

In [ ]:
X=data.drop(columns=['price'])
y=data['price']

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression,Lasso,Ridge
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import r2_score

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=0)

In [ ]:
print(X_train.shape)
print(X_test.shape)

(5917, 4)
(1480, 4)


**APPLY LINEAR REGRESSION**

In [ ]:
column_trans=make_column_transformer((OneHotEncoder(sparse_output=False),['location']),remainder='passthrough')

In [ ]:
scaler=StandardScaler()

In [ ]:
lr=LinearRegression()

In [ ]:
pipe=make_pipeline(column_trans,scaler,lr)

In [ ]:
pipe.fit(X_train,y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('columntransformer',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('onehotencoder',
                                                  OneHotEncoder(sparse_output=False),
                                                  ['location'])])),
                ('standardscaler', StandardScaler()),
                ('linearregression', LinearRegression())])

In [ ]:
y_pred_lr=pipe.predict(X_test)

In [ ]:
r2_score(y_test,y_pred_lr)

0.8441979436579821

In [ ]:
import pickle
pickle.dump(pipe,open('LinearRegressionModel_MINI_PRJCT.pkl','wb'))